In [ ]:
# Imports
from typing import Literal, List, Union, Optional, Annotated, Tuple, Callable
from collections import defaultdict
import time
import re

# Dependencies
import requests
import pandas as pd
import mwparserfromhell
from pydantic import BaseModel, Field

# Library
from classes.pipeline import VanguardClassifier, VanguardStorage, VanguardScrapper, VanguardParser, MediaWikiAPI, VanguardPipeline
from classes.cards import ScrapCard
from src.utils import classifier

# Definition
JSONType = dict[str]
header = {
	"User-Agent": "VanguardScrapper/1.0 (Python; contact: kmarrero1993@gmail.com)"
}
first_param = {
	"action": "parse",
	"page": "List of Cardfight!! Vanguard Booster Sets",
	"format": "json"
}


In [ ]:
# Url Parser
web = MediaWikiAPI()
pipeline = VanguardPipeline(
	VanguardScrapper(web),
	VanguardParser(),
	VanguardClassifier(),
	VanguardStorage()
)
curl = pipeline.scrapper.api.get(params=first_param, headers=header)
crude_data = pipeline.scrapper.obtain_links(curl)
crude_consults = pipeline.parser.separate_urls(crude_data)
consults = pipeline.parser.make_consults("get", crude_consults)
pipeline.parser.remove_from_list(crude_consults, [
	"Lyrical Monasterio", "The Mask Collection"
])
pipeline.parser.remove_from_list(crude_data, [
	"G Booster Set 8: Collezione del Combattente Vol.1",
	"G Booster Set 9: Collezione del Combattente Vol.2",
	"Thailand Booster Set 1: The Mask Collection",
	"G Booster Set 7: Giudizio delle Lame Splendenti",
	"G Booster Set 1: Trascendenza Interdimensionale"
])
classifier.process_items(crude_data, pipeline.classifier, pipeline.storage)
classifier.sort_storage_list(["LB", "G"], pipeline.classifier, pipeline.storage)
dz_consults = pipeline.parser.make_consults("consult", pipeline.storage.dz)
time.sleep(2)

curl = pipeline.scrapper.api.get(params=dz_consults[0], headers=header)
wikitex = pipeline.scrapper.obtain_wikitex(curl)

In [ ]:
x = mwparserfromhell.parse(wikitex)

In [ ]:
lst = []
for i in x.filter_templates():
	if ("CardList" in i.name):
		lst.append(i)

In [ ]:
pipeline.parser.remove_from_list(lst, ["{{CardList/header/D}}", "{{CardList/footer}}"])

In [ ]:
int(str(lst[0].params[2]))

In [ ]:
type(lst[0].params[2].value)

In [ ]:
row = {{i: [
	str(z.value).strip()
	for z in lst[142].params
	if z.value is not None]
} for i in range(6)}

In [ ]:
row = [
        str(z.value).strip()
        for z in lst[142].params
        if z.value != ''
    ]

In [ ]:
row

In [ ]:
row = [
    str(z.value).strip()
    for z in lst[142].params
    if z.value is not None
]

In [ ]:
row = {
    i: str(lst[150].params[i].value)
    for i in range(len(lst[150].params))
    if lst[150].params[i].value is not None
}

In [ ]:
row

In [ ]:
range(len(lst[142].params))

In [ ]:
lst[150]

In [ ]:
db = pd.DataFrame(rows, columns=["code", "name", "grade", "nation", "type", "rarity", "misc"])

In [ ]:
db

In [ ]:
lst[1].params

In [ ]:
row = [str(p.value).strip() for p in lst[1].params]

In [ ]:
row

In [ ]:
for i in row:
	print(i)

In [ ]:
db = pd.DataFrame([row], columns=["code", "name", "grade", "nation", "type", "rarity"])

In [ ]:
db

In [ ]:
l

In [ ]:
db = pd.DataFrame(l)

In [ ]:
db

In [ ]:
curl = pipeline.scrapper.api.get(params=dz_consults[0], headers=header)
content = pipeline.scrapper.obtain_links(curl)
pipeline.parser.clean_trash_from_set(crude_consults, content, 4)
card_consult = pipeline.parser.make_consults("consult", content)
main_table = pipeline.scrapper.api.get(params=card_consult[0], headers=header)

In [ ]:
s = main_table["query"]["pages"]["2010193"]["revisions"][0]["slots"]["main"]["*"]

In [ ]:
type(curl["query"]["pages"]["2038658"]["revisions"][0]["slots"]["main"]["*"])

In [ ]:
pages = main_table.get("query", {}).get("pages", {})

In [ ]:
page = next(iter(pages.values()))

In [ ]:
c = mwparserfromhell.parse(s)

In [ ]:
for i in c.filter_templates():
	print(i)

In [ ]:
main_table

In [ ]:
x